# Preproccessing of the dataset

In [ ]:
# [1] Imports
%load_ext autoreload
%autoreload 2

import io
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

from egg_hunt.config import RAW_DATA, CACHE, IMG_SIZE, NUM_CLASSES, CLASS_NAMES
from egg_hunt.preprocessing import (load_split, load_image, crop_egg,
                                    preprocess_image, preprocess_batch,
                                    build_arrays, get_cropped_arrays)

# ---------------------------------------
# Loading the image data
# ---------------------------------------

In [ ]:
# [2] Load the data
train_df = load_split(RAW_DATA / "train/train_labels.json", RAW_DATA / "train/train_data")
test_df  = load_split(RAW_DATA / "test/test_labels.json",   RAW_DATA / "test/test_data")
print(train_df.shape, test_df.shape)
train_df.head(3)

In [ ]:
# [3] CLASS_NAMES in config.py must match the categories in the JSON
json_names = (train_df[["category_id", "category_name"]]
              .drop_duplicates().sort_values("category_id")["category_name"].tolist())
assert json_names == CLASS_NAMES, "CLASS_NAMES in config.py doesn't match the labels JSON"
print("✅ class names match")

# ---------------------------------------
# Visualisation
# ---------------------------------------

In [ ]:
# [4] Visualisation: original with bbox, training crop, and what the model sees
def show_with_box(row):
    img = load_image(row["path"])
    x, y, w, h = row["bbox"]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="blue", facecolor="none"))
    ax.axis("off")
    plt.show()

row = train_df.iloc[3290]
show_with_box(row)

crop = crop_egg(row)
print("crop size (not resized):", crop.size)
display(crop)
display(Image.fromarray(preprocess_image(crop)))

In [ ]:
# [5] preprocess_image output: shape and dtype
arr = preprocess_image(crop)
assert arr.shape == (IMG_SIZE, IMG_SIZE, 3), arr.shape
assert arr.dtype == np.uint8, arr.dtype
print("✅", arr.shape, arr.dtype, "min/max:", arr.min(), arr.max())

In [ ]:
# [6] Every input type an uploader might send gives the same result
tmp_path = RAW_DATA / "test_upload.jpg"
crop.save(tmp_path)

from_path  = preprocess_image(tmp_path)                       # Path
from_str   = preprocess_image(str(tmp_path))                  # string path
from_bytes = preprocess_image(tmp_path.read_bytes())          # raw bytes (what the API gets)
with open(tmp_path, "rb") as f:
    from_file = preprocess_image(f)                           # file object (UploadFile.file)
from_pil   = preprocess_image(Image.open(tmp_path))           # PIL image
from_array = preprocess_image(np.asarray(Image.open(tmp_path)))  # numpy array

for name, a in [("str", from_str), ("bytes", from_bytes), ("file", from_file),
                ("pil", from_pil), ("array", from_array)]:
    assert np.array_equal(from_path, a), f"{name} differs from path"
print("✅ all input types give identical arrays")

In [ ]:
# [7] Non-square upload: should be padded with black, not stretched
x, y, w, h = row["bbox"]
img = load_image(row["path"])
wide_crop = img.crop((x - w, y, x + 2 * w, y + h))    # deliberately 3:1 wide
print("wide crop size:", wide_crop.size)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(wide_crop);                                            axes[0].set_title("upload (non-square)")
axes[1].imshow(wide_crop.resize((IMG_SIZE, IMG_SIZE)));               axes[1].set_title("old: stretched")
axes[2].imshow(preprocess_image(wide_crop));                          axes[2].set_title("new: padded")
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
# [8] preprocess_batch
batch = preprocess_batch([tmp_path, crop, from_array])
assert batch.shape == (3, IMG_SIZE, IMG_SIZE, 3)
print("✅ batch shape", batch.shape)

In [ ]:
# [9] New build_arrays vs your existing cache (old code), on the first 200 rows
N = 600
X_new, y_new = build_arrays(train_df.head(N))

old = np.load(CACHE)
X_old, y_old = old["X_train"][:N], old["y_train"][:N]

assert X_new.shape == X_old.shape and y_new.shape == y_old.shape
assert np.array_equal(y_new, y_old), "labels differ!"
print("✅ labels identical")

diff = np.abs(X_new.astype(int) - X_old.astype(int))
per_img = diff.reshape(N, -1).max(axis=1)
print(f"identical images: {(per_img == 0).sum()}/{N}")
print(f"mean pixel diff: {diff.mean():.4f}   max pixel diff: {diff.max()}")

In [ ]:
# [10] If [9] shows differences: look at the worst one
i = int(per_img.argmax())
print("row", i, "| crop size:", crop_egg(train_df.iloc[i]).size)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(X_old[i]);                     axes[0].set_title("old cache")
axes[1].imshow(X_new[i]);                     axes[1].set_title("new code")
axes[2].imshow(diff[i].max(axis=-1), cmap="hot"); axes[2].set_title("difference")
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
# [11] get_cropped_arrays: respects cache_path and reloads from the cache
small_cache = RAW_DATA / "test_cache_small.npz"
if small_cache.exists():
    small_cache.unlink()

a = get_cropped_arrays(train_df.head(20), test_df.head(10), cache_path=small_cache)  # builds
assert small_cache.exists(), "cache_path was ignored"
b = get_cropped_arrays(train_df.head(20), test_df.head(10), cache_path=small_cache)  # loads
assert all(np.array_equal(x, y) for x, y in zip(a, b))
print("✅ cache built at cache_path and reloaded identically:", [x.shape for x in b])

In [ ]:
# [12] Clean up test files
tmp_path.unlink(missing_ok=True)
small_cache.unlink(missing_ok=True)